# SHIPIT Agent: Gemma on Amazon Bedrock — an agent with tools

Google's **Gemma** runs on **Amazon Bedrock**, and in SHIPIT you reach it through the
one class you already use for Bedrock — **`BedrockChatLLM`**. Pass a Gemma model id
and it routes to the right endpoint for you; there is no new class to learn.

- **Gemma 4** — `google.gemma-4-31b` (dense, best quality), `google.gemma-4-26b-a4b`
  (mixture-of-experts, fast), `google.gemma-4-e2b` (small, cheap). Gemma 4 does
  **native function calling**, so it drives agents. On Bedrock it is served through
  the **OpenAI-compatible `bedrock-mantle`** endpoint (not the Converse API), and
  `BedrockChatLLM` delegates there transparently.
- **Gemma 3** — `google.gemma-3-27b-it` / `-12b-it` / `-4b-it`: chat via the
  standard Bedrock **Converse** API.

Regions at launch: **us-east-1, us-east-2, us-west-2, eu-central-1**.

Everything here runs **offline** — no Bedrock key. Section 3 injects a fake `openai`
module (the same pattern as `tests/test_gemma_bedrock.py`) so the *full* agentic loop
runs end to end with zero credentials. The one live cell is guarded behind an env
check.

In [1]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1 · Transparent routing

Construct `BedrockChatLLM` with a Gemma 4 id. Construction is **side-effect free** —
the `OpenAI` client is only built inside `.complete()`, so no key is needed just to
inspect the routing. A `google.gemma-4-*` id gets a `_mantle_delegate` pointed at the
mantle base URL; a Claude id has `_mantle_delegate is None` and uses Converse.

In [2]:
from shipit_agent.llms import BedrockChatLLM, BedrockGemmaChatLLM

# Gemma 4 → routed to the OpenAI-compatible bedrock-mantle endpoint.
gemma = BedrockChatLLM(model="google.gemma-4-31b", region="us-east-1")
print("delegate type :", type(gemma._mantle_delegate).__name__)
print("delegate model:", gemma._mantle_delegate.model)
print("base_url      :", gemma._mantle_delegate.client_kwargs["base_url"])

assert isinstance(gemma._mantle_delegate, BedrockGemmaChatLLM)
assert (
    gemma._mantle_delegate.client_kwargs["base_url"]
    == "https://bedrock-mantle.us-east-1.api.aws/openai/v1"
)

delegate type : BedrockGemmaChatLLM
delegate model: google.gemma-4-31b
base_url      : https://bedrock-mantle.us-east-1.api.aws/openai/v1


A Claude id (or any non-Gemma-4 model) has **no** mantle delegate — it takes the
normal Bedrock Converse path. The Gemma 3 ids behave the same way (chat via Converse).

In [3]:
claude = BedrockChatLLM(model="bedrock/anthropic.claude-3-5-sonnet-20240620-v1:0")
gemma3 = BedrockChatLLM(model="bedrock/google.gemma-3-27b-it")
print("claude  ._mantle_delegate:", claude._mantle_delegate)
print("gemma-3 ._mantle_delegate:", gemma3._mantle_delegate)
assert claude._mantle_delegate is None
assert gemma3._mantle_delegate is None

# The region also comes from AWS_REGION_NAME / the region kwarg.
eu = BedrockChatLLM(model="google.gemma-4-26b-a4b", aws_region_name="eu-central-1")
print("eu-central-1 base_url    :", eu._mantle_delegate.client_kwargs["base_url"])

claude  ._mantle_delegate: None
gemma-3 ._mantle_delegate: None
eu-central-1 base_url    : https://bedrock-mantle.eu-central-1.api.aws/openai/v1


## 2 · An agent with tools — the full loop, offline

Now the real thing: a Gemma 4 agent that **calls a tool**. We define an `add(a, b)`
`FunctionTool`, build an `Agent` around `BedrockChatLLM(model="google.gemma-4-31b")`,
then inject a fake `openai` module so the mantle delegate's `.complete()` returns
scripted responses:

- **turn 1** → a native tool call `add(a=2, b=3)`,
- **turn 2** → the final text answer.

This is exactly the mock from `tests/test_gemma_bedrock.py`. Because
`OpenAIChatLLM.complete()` imports `openai` *inside* the method, assigning
`sys.modules["openai"]` before `agent.run()` is enough — no real endpoint is hit.

In [4]:
import sys
import types
from types import SimpleNamespace as ns


def install_fake_openai(*responses):
    """Inject a fake `openai` module returning the given responses in order."""
    calls = {"i": 0}

    class _Completions:
        def create(self, **_kwargs):
            # Clamp so extra turns keep returning the last (text) response —
            # the agent loop always terminates cleanly.
            r = responses[min(calls["i"], len(responses) - 1)]
            calls["i"] += 1
            return r

    class _OpenAI:
        def __init__(self, **_kwargs):
            self.chat = ns(completions=_Completions())

    fake = types.ModuleType("openai")
    fake.OpenAI = _OpenAI
    sys.modules["openai"] = fake


def tool_call_response(name, arguments):
    msg = ns(
        content="",
        tool_calls=[ns(id="c1", function=ns(name=name, arguments=arguments))],
        reasoning_content=None,
    )
    return ns(choices=[ns(message=msg)], usage=None)


def text_response(text):
    msg = ns(content=text, tool_calls=[], reasoning_content=None)
    return ns(choices=[ns(message=msg)], usage=None)

Define the tool and the agent, script the two turns, then run.

In [5]:
from typing import Any
from shipit_agent import Agent, FunctionTool

ran: list[str] = []


def add(a: int, b: int, **_: Any) -> str:
    """Add two numbers."""
    ran.append("add")
    return str(a + b)


# Turn 1: Gemma 4 emits a native tool call. Turn 2: it answers in text.
install_fake_openai(
    tool_call_response("add", '{"a": 2, "b": 3}'),
    text_response("The answer is 5."),
)

agent = Agent(
    llm=BedrockChatLLM(model="google.gemma-4-31b", region="us-east-1"),
    tools=[FunctionTool.from_callable(add, name="add")],
    auto_use_skills=False,
)

result = agent.run("What is 2 + 3?")

print("output    :", result.output)
print("tool ran  :", ran)          # ['add'] — the tool actually executed
assert ran == ["add"]
assert "answer is 5" in result.output

output    : The answer is 5.
tool ran  : ['add']


That is the complete agentic loop: **model → native tool call → tool executes →
model → final answer**, driven by Gemma 4 through `BedrockChatLLM`, with **zero
credentials**. The `ran == ["add"]` assertion proves the tool really ran (it wasn't
the model just talking about adding).

## 3 · Running it live

To run against real Bedrock, create a **Bedrock API key** and export it:

1. AWS console → **Amazon Bedrock** → **API keys** → create a key.
2. Export it plus a supported region:

   ```bash
   export AWS_BEARER_TOKEN_BEDROCK=...        # the Bedrock API key
   export AWS_REGION_NAME=us-east-1           # or us-east-2 / us-west-2 / eu-central-1
   ```

Then it is a one-liner — the same `BedrockChatLLM`, this time with the full built-in
tool catalogue via `Agent.with_builtins`. The cell below is guarded on
`AWS_BEARER_TOKEN_BEDROCK`, so it is skipped offline and the notebook stays clean.

In [6]:
import os

if os.getenv("AWS_BEARER_TOKEN_BEDROCK"):
    # Real call — Gemma 4 via the bedrock-mantle endpoint, with built-in tools.
    live_llm = BedrockChatLLM(model="google.gemma-4-31b")   # region from AWS_REGION_NAME
    live_agent = Agent.with_builtins(
        llm=live_llm,
        tools=[FunctionTool.from_callable(add, name="add")],
    )
    live = live_agent.run("What is 2 + 3? Use a tool if it helps.")
    print(live.output)
else:
    print("Set AWS_BEARER_TOKEN_BEDROCK (a Bedrock API key) to run this live.")
    print("AWS console -> Amazon Bedrock -> API keys -> create, then:")
    print("  export AWS_BEARER_TOKEN_BEDROCK=...  AWS_REGION_NAME=us-east-1")
    print("Skipping the live call (offline).")

Set AWS_BEARER_TOKEN_BEDROCK (a Bedrock API key) to run this live.
AWS console -> Amazon Bedrock -> API keys -> create, then:
  export AWS_BEARER_TOKEN_BEDROCK=...  AWS_REGION_NAME=us-east-1
Skipping the live call (offline).


## 4 · Provider note

The agent code above is **provider-agnostic** — `Agent`, `FunctionTool`, and
`agent.run(...)` never mention Gemma or Bedrock. Gemma is just a **model string**.
Swap the `llm=` and the exact same tool-using agent runs on any provider:

```python
from shipit_agent.llms import (
    BedrockChatLLM,      # Gemma 4 / Gemma 3 / Claude / Nova ... on Bedrock
    AnthropicChatLLM,    # Claude direct
    OpenAIChatLLM,       # GPT
    GeminiChatLLM,       # Gemini
)

llm = BedrockChatLLM(model="google.gemma-4-31b")   # <- Gemma on Bedrock
# llm = AnthropicChatLLM(model="claude-opus-4-1")  # <- same agent, different model
agent = Agent.with_builtins(llm=llm, tools=[...])
```

### Recap

- One class, `BedrockChatLLM`, covers all of Gemma on Bedrock — pass the id and it
  routes: **Gemma 4 → mantle (native tool use)**, **Gemma 3 / Claude → Converse**.
- Construction is side-effect free; `_mantle_delegate` shows where a Gemma-4 id lands.
- The full agentic loop (tool call → execute → answer) runs offline by injecting a
  fake `openai` module — the same pattern the test suite uses.
- Going live is one Bedrock API key + a region away, and the agent code doesn't
  change one line to move between providers.